YOLO → JSON → LangChain → Human readable → Dashboard

In [4]:
def generate_alert(event: dict) -> dict:
    """
    Takes PPE violation JSON and returns human-readable alert + structured output
    """

    zone = event.get("zone", "Unknown Zone")
    timestamp = event.get("timestamp", "Unknown Time")
    types = event.get("type", [])
    confidence = event.get("confidence", 0)

    violations = []

    # Map internal labels → human readable
    mapping = {
        "no_helmet": "helmet",
        "no_vest": "safety vest",
        "no_gloves": "gloves",
        "no_boots": "safety boots"
    }

    for t in types:
        if t in mapping:
            violations.append(mapping[t])

    # No violation case
    if not violations:
        message = f"All safety equipment is properly worn in {zone} at {timestamp}."
        severity = "low"

    # Single violation
    elif len(violations) == 1:
        message = f"Worker in {zone} is not wearing a {violations[0]} at {timestamp}."
        severity = "medium"

    # Multiple violations
    else:
        items = ", ".join(violations[:-1]) + " and " + violations[-1]
        message = f"Worker in {zone} is not wearing {items} at {timestamp}."
        severity = "high"

    # Adjust severity if helmet missing (critical rule)
    if "no_helmet" in types:
        severity = "high"

    return {
        "message": message,
        "severity": severity,
        "confidence": round(confidence, 2),
        "zone": zone,
        "timestamp": timestamp,
        "violations": types
    }

In [5]:
if __name__ == "__main__":
    
    test_event = {
        "type": ["no_helmet", "no_gloves"],
        "confidence": 0.87,
        "timestamp": "10:32",
        "zone": "Zone A"
    }

    result = generate_alert(test_event)

    print(result["message"])
    print(result)

Worker in Zone A is not wearing helmet and gloves at 10:32.
{'message': 'Worker in Zone A is not wearing helmet and gloves at 10:32.', 'severity': 'high', 'confidence': 0.87, 'zone': 'Zone A', 'timestamp': '10:32', 'violations': ['no_helmet', 'no_gloves']}
